# Deep Learning Methods - Code

This notebook will demonstrate how to use the transformers library from Hugging Face to fine-tune BART and T5 for our review summarization task. Our steps include:

- **Load and process our data:** Create a dataset class that includes tokenized inputs and targets
- **Define training logic:** Write training functions for BART and T5
- **Generate summaries:** Use our fine-tuned models to generate abstractive summaries and compare with gold summaries

In [2]:
# Install necessary libraries

!pip install -q rouge_score evaluate bert_score

In [3]:
# Import necessary libraries

import pandas as pd
import numpy as np
import os
import json
import random
import torch
from torch.utils.data import Dataset
from transformers import BartForConditionalGeneration, BartTokenizer, T5ForConditionalGeneration, T5Tokenizer
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import evaluate
import textwrap
from sklearn.model_selection import train_test_split
import ipywidgets

from rouge_score import rouge_scorer
from bert_score import score as bert_score

import gc

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Set seeds for reproducibility

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [5]:
# Models are fine-tuned on GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load our processed data

train_df = pd.read_json('data/amasum_train_processed.json', orient='records', lines=True)
test_df = pd.read_json('data/amasum_test_processed.json', orient='records', lines=True)

In [7]:
# Create validation dataset (can be moved to EDA section)

train_df, valid_df = train_test_split(train_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

## Summarization

The following section will cover how we import pretrained BART and T5 models and fine-tune them with our data. Below we define which models we'll be using, load pretrained instances of them, and load their tokenizers:

In [8]:
bart_model_name = 'facebook/bart-large-cnn'
bart_model = BartForConditionalGeneration.from_pretrained(bart_model_name)
bart_tokenizer = BartTokenizer.from_pretrained(bart_model_name)

In [9]:
t5_model_name = 't5-base'
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


### Preparing our data

We will start by defining our inputs and targets for the models - this involves tokenizing our review subsets and gold summaries, then truncating and padding them to normalize their sizes.

In [10]:
# Tokenize and process our inputs and targets

def create_dataset(df, tokenizer, model):
    all_reviews = df['cluster_reviews'].apply(lambda x: " [SEP] ".join(x))

    if model == 'BART': # Short prefixes can help guide the model
      verdict_reviews = ["Summarize: " + r for r in all_reviews]
      pros_reviews = ["List the POSITIVE aspects: " + r for r in all_reviews]
      cons_reviews = ["List the NEGATIVE aspects: " + r for r in all_reviews]
    elif model == 'T5': # T5 was trained with task prefixes, so more explicit prompts help
      verdict_reviews = ["summarize the following customer product reviews: " + r for r in all_reviews]
      pros_reviews = ["generate a list of the positive aspects of the product based on these reviews: " + r for r in all_reviews]
      cons_reviews = ["generate a list of the negative aspects of the product based on these reviews: " + r for r in all_reviews]


    verdicts = [v for v in df['verdict']]
    pros = [", ".join(p) for p in df['pros']] # Some pros are lists, so we join them into a string
    cons = [", ".join(c) for c in df['cons']] # Same for cons

    # Tokenize inputs and targets (padding will be handled later by the DataCollator)
    verdict_inputs = tokenizer(verdict_reviews, truncation=True, padding=False, max_length=1024)
    verdict_targets = tokenizer(verdicts, truncation=True, padding=False, max_length=80)

    pros_inputs = tokenizer(pros_reviews, truncation=True, padding=False, max_length=1024)
    pros_targets = tokenizer(pros, truncation=True, padding=False, max_length=80)

    cons_inputs = tokenizer(cons_reviews, truncation=True, padding=False, max_length=1024)
    cons_targets = tokenizer(cons, truncation=True, padding=False, max_length=80)

    return {
        'verdict_inputs': verdict_inputs,
        'verdict_targets': verdict_targets,
        'pros_inputs': pros_inputs,
        'pros_targets': pros_targets,
        'cons_inputs': cons_inputs,
        'cons_targets': cons_targets
    }

In [11]:
# Define a custom Dataset class

class SummarizationDataset(Dataset):
    def __init__(self, inputs, targets, tokenizer):
        self.input_ids = inputs.input_ids
        self.attention_mask = inputs.attention_mask
        self.labels = []

        for label_seq in targets.input_ids:
            # Replace pad_token_id with -100
            masked_labels = [
                token_id if token_id != tokenizer.pad_token_id else -100
                for token_id in label_seq
            ]
            self.labels.append(masked_labels)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

## Model Training

Before training, we define a "compute_metrics" function that evaluates model performance during training. The only difference between these two functions is the tokenizer, which we can't pass as an argument during training.

In [12]:
# Define a compute_metrics function

rouge_metric = evaluate.load("rouge")

def compute_metrics_bart(eval_pred):

    predictions, labels = eval_pred

    # Unpack logits if tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert logits to token IDs if needed
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Convert labels: replace -100 with pad_token_id for decoding
    labels = np.where(labels != -100, labels, bart_tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = bart_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE scores
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Optional: Return only F1 scores (most commonly used)
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }


def compute_metrics_t5(eval_pred):

    predictions, labels = eval_pred

    # Unpack logits if tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert logits to token IDs if needed
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Convert labels: replace -100 with pad_token_id for decoding
    labels = np.where(labels != -100, labels, t5_tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = t5_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE scores
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Optional: Return only F1 scores (most commonly used)
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

Next, we'll define the function for fine-tuning our models. Tokenizers and training arguments will vary by model, but the overall training logic will be the same for both BART and T5. We will train three separate model instances - one for each target (verdict, pros, and cons) as these are distinct tasks.

In [13]:
# Training function

def train_model(train_inputs, valid_inputs, train_targets, valid_targets, target_type, training_args, compute_metrics_func, model):

    if model == 'BART':
      task_model = BartForConditionalGeneration.from_pretrained(bart_model_name)
      tokenizer = BartTokenizer.from_pretrained(bart_model_name)
    elif model == 'T5':
      task_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
      tokenizer = T5Tokenizer.from_pretrained(t5_model_name)


    train_dataset = SummarizationDataset(train_inputs, train_targets, tokenizer)
    eval_dataset = SummarizationDataset(valid_inputs, valid_targets, tokenizer)

    training_args = TrainingArguments(
        output_dir = f'./{target_type}_model',
        **training_args
    )

    print(f"Training {target_type} model...")

    trainer = Trainer(
        model=task_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics_func,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=task_model),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    trainer.train()

    return task_model

::: {dropdown} Code Explanation

```python
if model == 'BART':
  task_model = BartForConditionalGeneration.from_pretrained(bart_model_name)
  tokenizer = BartTokenizer.from_pretrained(bart_model_name)
elif model == 'T5':
  task_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
  tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
```
This block initializes the model and tokenizer based on the input model name. It supports either BART or T5, both commonly used for text summarization tasks.

```python
train_dataset = SummarizationDataset(train_inputs, train_targets, tokenizer)
eval_dataset = SummarizationDataset(valid_inputs, valid_targets, tokenizer)
```
The training and validation datasets are prepared using a custom `SummarizationDataset` class, which tokenizes the input-target pairs for supervised training.

```python
training_args = TrainingArguments(
    output_dir = f'./{target_type}_model',
    **training_args
)
```
This sets up the training arguments using Hugging Face's `TrainingArguments` class, dynamically assigning the output directory and unpacking additional configurations provided via `training_args`.

```python
print(f"Training {target_type} model...")
```
A simple logging statement to indicate the start of model training for a specific target type (e.g., "verdict", "pros", or "cons").

```python
trainer = Trainer(
    model=task_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_func,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=task_model),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()
```
This initializes and runs Hugging Face's `Trainer`, handling training and evaluation. It uses early stopping to prevent overfitting and a data collator to batch inputs for sequence-to-sequence learning.

```python
return task_model
```
After training, the fine-tuned model is returned for downstream use, such as inference or further evaluation.

:::


Finally, we define a function to generate summaries from our fine-tuned models. It processing batches of input token tensors through separate models for each component, using task-specific generation settings.

In [ ]:
def generate_summaries(
    verdict_inputs, pros_inputs, cons_inputs,
    verdict_model, pros_model, cons_model,
    tokenizer,
    max_length=64,
    batch_size=4,
):
    def process_in_batches(inputs, model, model_name, task_type):
        model.to(device)
        all_decoded = []

        print(f"Processing {model_name}...")

        for i in range(0, len(inputs["input_ids"]), batch_size):
            batch_inputs = {
                "input_ids": inputs["input_ids"][i:i+batch_size],
                "attention_mask": inputs["attention_mask"][i:i+batch_size]
            }

            examples = [
                {"input_ids": ids, "attention_mask": mask}
                for ids, mask in zip(batch_inputs["input_ids"], batch_inputs["attention_mask"])
            ]

            padded = tokenizer.pad(examples, padding=True, return_tensors="pt")
            padded = {k: v.to(device) for k, v in padded.items()}

            with torch.no_grad():
                # Task-specific generation parameters
                if task_type == "verdict":
                    outputs = model.generate(
                        **padded,
                        max_length=max_length,
                        min_length=20,
                        num_beams=4, 
                        do_sample=False,
                        early_stopping=True,

                        # Anti-Repetition
                        no_repeat_ngram_size=3,         # Prevent 3-gram repetition
                        repetition_penalty=1.2,        # Penalize repeated tokens
                        length_penalty=1.0,            # Neutral length preference

                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                else:  # pros/cons
                    outputs = model.generate(
                        **padded,
                        max_length=max_length,
                        min_length=10,
                        num_beams=3,
                        do_sample=True,
                        temperature=0.8,
                        top_p=0.9,
                        early_stopping=True,

                        no_repeat_ngram_size=2,         # Prevent 2-gram repetition
                        repetition_penalty=1.3,        # Pros/cons need more diversity
                        length_penalty=0.8,            # Conciseness for pros/cons

                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )

            batch_decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_decoded.extend(batch_decoded)

            del outputs, padded
            torch.cuda.empty_cache()

        model.to('cpu')
        torch.cuda.empty_cache()

        return all_decoded

    # Process each model type with task-specific parameters
    verdicts = process_in_batches(verdict_inputs, verdict_model, "verdicts", "verdict")
    pros = process_in_batches(pros_inputs, pros_model, "pros", "pros")
    cons = process_in_batches(cons_inputs, cons_model, "cons", "cons")

    summaries = [
        {"verdict": v.strip(), "pros": p.strip(), "cons": c.strip()}
        for v, p, c in zip(verdicts, pros, cons)
    ]

    return summaries

::: {dropdown} Code Explanation

```python
def generate_summaries(
    verdict_inputs, pros_inputs, cons_inputs,
    verdict_model, pros_model, cons_model,
    tokenizer,
    max_length=64,
    batch_size=4,
):
    def process_in_batches(inputs, model, model_name, task_type):
        model.to(device)
        all_decoded = []

        print(f"Processing {model_name}...")
```

The main function `generate_summaries` generates summaries for verdicts, pros, and cons using separate models.  
It defines a helper function `process_in_batches` that moves the model to the GPU (`device`) and initializes storage for decoded outputs, also printing the task being processed.

```python
        for i in range(0, len(inputs["input_ids"]), batch_size):
            batch_inputs = {
                "input_ids": inputs["input_ids"][i:i+batch_size],
                "attention_mask": inputs["attention_mask"][i:i+batch_size]
            }

            examples = [
                {"input_ids": ids, "attention_mask": mask}
                for ids, mask in zip(batch_inputs["input_ids"], batch_inputs["attention_mask"])
            ]

            padded = tokenizer.pad(examples, padding=True, return_tensors="pt")
            padded = {k: v.to(device) for k, v in padded.items()}
```

The loop processes input data in batches to avoid memory overload.  
It slices out a batch of token IDs and attention masks, prepares them as dictionaries, pads them for uniform length, and transfers them to the GPU.

```python
            with torch.no_grad():
                if task_type == "verdict":
                    outputs = model.generate(
                        **padded,
                        max_length=max_length,
                        min_length=20,
                        num_beams=4, 
                        do_sample=False,
                        early_stopping=True,
                        no_repeat_ngram_size=3,
                        repetition_penalty=1.2,
                        length_penalty=1.0,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                else:  # pros/cons
                    outputs = model.generate(
                        **padded,
                        max_length=max_length,
                        min_length=10,
                        num_beams=3,
                        do_sample=True,
                        temperature=0.8,
                        top_p=0.9,
                        early_stopping=True,
                        no_repeat_ngram_size=2,
                        repetition_penalty=1.3,
                        length_penalty=0.8,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
```

Text generation parameters differ by task type:  
- **Verdicts**: Beam search with stricter repetition control, fixed length preference, no sampling.  
- **Pros/Cons**: More randomness via temperature and top-p sampling, looser repetition control, shorter length preference for concise outputs.

```python
            batch_decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_decoded.extend(batch_decoded)

            del outputs, padded
            torch.cuda.empty_cache()

        model.to('cpu')
        torch.cuda.empty_cache()

        return all_decoded
```

Generated outputs are decoded into plain text, stored, and memory is freed after each batch to prevent GPU overload. Once processing is complete, the model is moved back to CPU.

```python
    verdicts = process_in_batches(verdict_inputs, verdict_model, "verdicts", "verdict")
    pros = process_in_batches(pros_inputs, pros_model, "pros", "pros")
    cons = process_in_batches(cons_inputs, cons_model, "cons", "cons")

    summaries = [
        {"verdict": v.strip(), "pros": p.strip(), "cons": c.strip()}
        for v, p, c in zip(verdicts, pros, cons)
    ]

    return summaries
```

The helper function is run separately for each input set and model.  
The results are combined into dictionaries containing the cleaned verdict, pros, and cons for each example, which are returned as a list of summaries.

:::


### BART

We will fine-tune BART first. With our model and tokenizer defined, we need to create the datasets for our model, define the training arguments, run the training function, then generate summaries.

In [16]:
bart_train_dataset = create_dataset(train_df, bart_tokenizer, 'BART')
bart_valid_dataset = create_dataset(valid_df, bart_tokenizer, 'BART')
bart_test_dataset = create_dataset(test_df, bart_tokenizer, 'BART')

In [ ]:
bart_training_args = {
    "seed": 42, # Reproducibility
    "num_train_epochs": 3,  # Train for 3 full passes over the dataset
    "per_device_train_batch_size": 4,  # Small batch size to avoid OOM errors
    "per_device_eval_batch_size": 8,  # Match eval batch size to training for memory consistency
    "learning_rate": 1e-5,  # Low LR suitable for fine-tuning pretrained models
    "warmup_ratio": 0.1,  # Gradually increase LR over the first 10% of steps
    "weight_decay": 0.01,  # Apply regularization to reduce overfitting
    "save_steps": 200,  # Save model every 200 steps during training
    "logging_steps": 25,  # Log training metrics every 25 steps
    "eval_steps": 50,  # Evaluate model every 50 steps to monitor progress
    "save_total_limit": 3,  # Keep only the 3 most recent saved checkpoints
    "eval_strategy": "steps",  # Evaluate every N steps instead of each epoch
    "metric_for_best_model": "eval_loss",  # Use eval loss to determine the best checkpoint
    "greater_is_better": False,  # Lower eval loss is better
    "load_best_model_at_end": True,  # Reload best model after training ends
    "fp16": False,  # Disable mixed precision training for stability or hardware compatibility
    "gradient_accumulation_steps": 2,  # Accumulate gradients to simulate a larger batch size
    "dataloader_pin_memory": True,  # Speed up data transfer to GPU
    "report_to": "none",  # Disable integration with external logging services like WandB
    "label_smoothing_factor": 0.1,      # Prevents overconfident predictions
    "dataloader_num_workers": 0
}

In [18]:
bart_verdict_model = train_model(bart_train_dataset['verdict_inputs'],
                           bart_valid_dataset['verdict_inputs'],
                           bart_train_dataset['verdict_targets'],
                           bart_valid_dataset['verdict_targets'],
                           'verdict',
                           bart_training_args,
                           compute_metrics_bart,
                           'BART')

Training verdict model...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,4.180200,4.020080,0.363181,0.106260,0.326521
100,4.072500,3.975532,0.368314,0.111102,0.333304
150,3.773400,3.962572,0.373472,0.114133,0.335122
200,3.778100,3.942206,0.375552,0.114074,0.338306
250,3.803000,3.938181,0.376050,0.117974,0.340189
300,3.584800,3.942450,0.373983,0.117672,0.340036
350,3.624000,3.941784,0.372502,0.117444,0.338721


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


In [19]:
bart_pros_model = train_model(bart_train_dataset['pros_inputs'],
                        bart_valid_dataset['pros_inputs'],
                        bart_train_dataset['pros_targets'],
                        bart_valid_dataset['pros_targets'],
                        'pros',
                        bart_training_args,
                        compute_metrics_bart,
                        'BART')

Training pros model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,4.636800,4.492434,0.337667,0.075441,0.291279
100,4.483900,4.411364,0.342769,0.080290,0.296767
150,4.211900,4.396379,0.346487,0.078907,0.299424
200,4.271500,4.378372,0.345698,0.080202,0.298404
250,4.221800,4.367940,0.345800,0.082107,0.300345
300,4.048800,4.373902,0.342960,0.081406,0.298695
350,4.071200,4.367343,0.344768,0.081919,0.300033


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


In [20]:
bart_cons_model = train_model(bart_train_dataset['cons_inputs'],
                        bart_valid_dataset['cons_inputs'],
                        bart_train_dataset['cons_targets'],
                        bart_valid_dataset['cons_targets'],
                        'cons',
                        bart_training_args,
                        compute_metrics_bart,
                        'BART')

Training cons model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,4.510300,4.447452,0.313038,0.078361,0.284820
100,4.314900,4.351159,0.322247,0.082630,0.294406
150,4.016700,4.340896,0.319595,0.079436,0.294914
200,3.988100,4.338297,0.315952,0.080352,0.290068
250,3.939500,4.330439,0.322997,0.081519,0.297748
300,3.743100,4.325093,0.322264,0.085289,0.298621
350,3.733500,4.336628,0.317017,0.081515,0.293462


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


In [21]:
bart_generated = generate_summaries(
            bart_test_dataset['verdict_inputs'],
            bart_test_dataset['pros_inputs'],
            bart_test_dataset['cons_inputs'],
            bart_verdict_model,
            bart_pros_model,
            bart_cons_model,
            bart_tokenizer
)

Processing verdicts...
Processing pros...
Processing cons...


## T5

We follow the same process for T5:

In [22]:
t5_train_dataset = create_dataset(train_df, t5_tokenizer, 'T5')
t5_valid_dataset = create_dataset(valid_df, t5_tokenizer, 'T5')
t5_test_dataset = create_dataset(test_df, t5_tokenizer, 'T5')

In [ ]:
t5_training_args = {
    "seed": 42, # Reproducibility
    "num_train_epochs": 3,  # Train for 3 full passes over the dataset
    "per_device_train_batch_size": 4,  # Small batch size to avoid OOM errors
    "per_device_eval_batch_size": 8,  # Match eval batch size to training for memory consistency
    "learning_rate": 1e-5,  # Low LR suitable for fine-tuning pretrained models
    "warmup_ratio": 0.1,  # Gradually increase LR over the first 10% of steps
    "weight_decay": 0.01,  # Apply regularization to reduce overfitting
    "save_steps": 200,  # Save model every 200 steps during training
    "logging_steps": 25,  # Log training metrics every 25 steps
    "eval_steps": 50,  # Evaluate model every 50 steps to monitor progress
    "save_total_limit": 3,  # Keep only the 3 most recent saved checkpoints
    "eval_strategy": "steps",  # Evaluate every N steps instead of each epoch
    "metric_for_best_model": "eval_loss",  # Use eval loss to determine the best checkpoint
    "greater_is_better": False,  # Lower eval loss is better
    "load_best_model_at_end": True,  # Reload best model after training ends
    "fp16": False,  # Disable mixed precision training for stability or hardware compatibility
    "gradient_accumulation_steps": 2,  # Accumulate gradients to simulate a larger batch size
    "dataloader_pin_memory": True,  # Speed up data transfer to GPU
    "report_to": "none",  # Disable integration with external logging services like WandB
    "label_smoothing_factor": 0.1,      # Prevents overconfident predictions
    "dataloader_num_workers": 0
}

In [24]:
t5_verdict_model = train_model(t5_train_dataset['verdict_inputs'],
                         t5_valid_dataset['verdict_inputs'],
                         t5_train_dataset['verdict_targets'],
                         t5_valid_dataset['verdict_targets'],
                         'verdict',
                         t5_training_args,
                         compute_metrics_t5,
                         'T5')

Training verdict model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,5.047600,4.677310,0.297196,0.074093,0.264821
100,4.588000,4.364609,0.330923,0.084481,0.297143
150,4.466900,4.263638,0.332738,0.088147,0.302894
200,4.388900,4.204455,0.326491,0.088323,0.297769
250,4.370800,4.174933,0.316724,0.083632,0.288257
300,4.323800,4.156750,0.316048,0.084060,0.287552
350,4.365500,4.149462,0.316577,0.084505,0.288070


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [25]:
t5_pros_model = train_model(t5_train_dataset['pros_inputs'],
                      t5_valid_dataset['pros_inputs'],
                      t5_train_dataset['pros_targets'],
                      t5_valid_dataset['pros_targets'],
                      'pros',
                      t5_training_args,
                      compute_metrics_t5,
                      'T5')

Training pros model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,5.583200,5.248425,0.317396,0.062623,0.269634
100,5.042300,4.779184,0.326028,0.064183,0.281179
150,4.823500,4.623443,0.327573,0.064437,0.286629
200,4.812000,4.558871,0.327502,0.066058,0.287065
250,4.695600,4.526033,0.328505,0.066629,0.287626
300,4.663400,4.512206,0.328339,0.066286,0.288296
350,4.675400,4.503953,0.327421,0.066103,0.287052


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [26]:
t5_cons_model = train_model(t5_train_dataset['cons_inputs'],
                      t5_valid_dataset['cons_inputs'],
                      t5_train_dataset['cons_targets'],
                      t5_valid_dataset['cons_targets'],
                      'cons',
                      t5_training_args,
                      compute_metrics_t5,
                      'T5')

Training cons model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,5.897800,5.631890,0.267656,0.051602,0.243305
100,5.223400,4.996595,0.279871,0.055619,0.251928
150,4.932600,4.749451,0.277619,0.056698,0.255715
200,4.782400,4.659109,0.265641,0.054922,0.247533
250,4.665600,4.620559,0.257036,0.054102,0.237987
300,4.691900,4.599750,0.254120,0.054920,0.234532
350,4.665000,4.589254,0.255572,0.056254,0.235982


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [27]:
t5_generated = generate_summaries(t5_test_dataset['verdict_inputs'],
                                  t5_test_dataset['pros_inputs'],
                                  t5_test_dataset['cons_inputs'],
                                  t5_verdict_model,
                                  t5_pros_model,
                                  t5_cons_model,
                                  t5_tokenizer
)

Processing verdicts...
Processing pros...
Processing cons...


## Evaluation

How do our generated summaries compare with the gold summaries?

In [28]:
gold_summaries = test_df[['verdict', 'pros', 'cons']].to_dict(orient='records')

In [29]:
# Define ROUGE evaluation functions

def evaluate_rouge_single(generated_summaries, gold_summaries):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for gen, gold in zip(generated_summaries, gold_summaries):
        scores = scorer.score(gold, gen)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    # Calculate averages
    avg_rouge1 = round(np.mean(rouge1_scores).item(), 4)
    avg_rouge2 = round(np.mean(rouge2_scores).item(), 4)
    avg_rougeL = round(np.mean(rougeL_scores).item(), 4)

    return {
        'rouge1': avg_rouge1,
        'rouge2': avg_rouge2,
        'rougeL': avg_rougeL
    }


def evaluate_rouge(generated, gold_data):
    # Extract generated parts
    gen_verdicts = [item['verdict'] for item in generated]
    gen_pros = [' '.join(item['pros']) for item in generated]
    gen_cons = [' '.join(item['cons']) for item in generated]

    # Extract gold parts
    gold_verdicts = [item['verdict'] for item in gold_data]
    gold_pros = [' '.join(item['pros']) for item in gold_data]
    gold_cons = [' '.join(item['cons']) for item in gold_data]

    # Evaluate each component
    verdict_rouge = evaluate_rouge_single(gen_verdicts, gold_verdicts)
    pros_rouge = evaluate_rouge_single(gen_pros, gold_pros)
    cons_rouge = evaluate_rouge_single(gen_cons, gold_cons)

    return {
        'verdict': verdict_rouge,
        'pros': pros_rouge,
        'cons': cons_rouge
    }

In [43]:
# Define BERTScore evaluation functions

def evaluate_bertscore_single(generated_summaries, gold_summaries, lang='en'):
    # Calculate BERTScore
    P, R, F1 = bert_score(generated_summaries, gold_summaries, lang=lang, verbose=False)

    return {
        'precision': round(P.numpy().mean().item(), 4),
        'recall': round(R.numpy().mean().item(), 4),
        'f1': round(F1.numpy().mean().item(), 4)
    }

def evaluate_bertscore(generated, gold_data, lang='en'):
    # Extract generated parts
    gen_verdicts = [item['verdict'] for item in generated]
    gen_pros = [' '.join(item['pros']) for item in generated]
    gen_cons = [' '.join(item['cons']) for item in generated]

    # Extract gold parts
    gold_verdicts = [item['verdict'] for item in gold_data]
    gold_pros = [' '.join(item['pros']) for item in gold_data]
    gold_cons = [' '.join(item['cons']) for item in gold_data]

    # Evaluate each component with BERTScore
    verdict_bert = evaluate_bertscore_single(gen_verdicts, gold_verdicts, lang)
    pros_bert = evaluate_bertscore_single(gen_pros, gold_pros, lang)
    cons_bert = evaluate_bertscore_single(gen_cons, gold_cons, lang)

    return {
        'verdict': verdict_bert,
        'pros': pros_bert,
        'cons': cons_bert
    }

### ROUGE Results

In [44]:
bart_rouge_results = evaluate_rouge(bart_generated, gold_summaries)

print("BART ROUGE Results:\n")
print(f"VERDICT:{bart_rouge_results['verdict']}")
print(f"PROS:{bart_rouge_results['pros']}")
print(f"CONS:{bart_rouge_results['cons']}")

BART ROUGE Results:

VERDICT:{'rouge1': 0.2146, 'rouge2': 0.041, 'rougeL': 0.1619}
PROS:{'rouge1': 0.0145, 'rouge2': 0.0001, 'rougeL': 0.0144}
CONS:{'rouge1': 0.0182, 'rouge2': 0.0001, 'rougeL': 0.0182}


In [45]:
t5_rouge_results = evaluate_rouge(t5_generated, gold_summaries)

print("T5 ROUGE Results:\n")
print(f"VERDICT:{t5_rouge_results['verdict']}")
print(f"PROS:{t5_rouge_results['pros']}")
print(f"CONS:{t5_rouge_results['cons']}")

T5 ROUGE Results:

VERDICT:{'rouge1': 0.2086, 'rouge2': 0.0346, 'rougeL': 0.1625}
PROS:{'rouge1': 0.0184, 'rouge2': 0.0001, 'rougeL': 0.0182}
CONS:{'rouge1': 0.0185, 'rouge2': 0.0, 'rougeL': 0.0185}


BART and T5 achieve comparable ROUGE results. ROUGE scores are very similar for generated verdicts, while T5's generated pros and cons seem to have slightly more n-gram overlap. These are slight improvements on our extractive models.

### BERTScore Results

In [46]:
bart_bertscore_results = evaluate_bertscore(bart_generated, gold_summaries)

print("BART BERTScore Results:\n")
print(f"VERDICT:{bart_bertscore_results['verdict']}")
print(f"PROS:{bart_bertscore_results['pros']}")
print(f"CONS:{bart_bertscore_results['cons']}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BART BERTScore Results:

VERDICT:{'precision': 0.8805, 'recall': 0.8739, 'f1': 0.8771}
PROS:{'precision': 0.5979, 'recall': 0.7786, 'f1': 0.6763}
CONS:{'precision': 0.6073, 'recall': 0.8027, 'f1': 0.6913}


In [47]:
t5_bertscore_results = evaluate_bertscore(t5_generated, gold_summaries)

print("T5 BERTScore Results:\n")
print(f"VERDICT:{t5_bertscore_results['verdict']}")
print(f"PROS:{t5_bertscore_results['pros']}")
print(f"CONS:{t5_bertscore_results['cons']}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


T5 BERTScore Results:

VERDICT:{'precision': 0.8809, 'recall': 0.8708, 'f1': 0.8757}
PROS:{'precision': 0.6067, 'recall': 0.7802, 'f1': 0.6825}
CONS:{'precision': 0.6125, 'recall': 0.8033, 'f1': 0.6949}


BERTScores are also very comparable across all summaries, with T5 again showing slightly better performance for generated pros and cons.

In [ ]:
# Save results

results_to_save = {
    "gold_summaries": gold_summaries,
    "bart_rouge_results": bart_rouge_results,
    "t5_rouge_results": t5_rouge_results,
    "bart_bertscore_results": bart_bertscore_results,
    "t5_bertscore_results": t5_bertscore_results,
    "bart_generated": bart_generated,
    "t5_generated": t5_generated
}

# Save each dictionary as a JSON file
for name, data in results_to_save.items():
    file_path = f"../results/{name}.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("All results saved.")

All results saved.
